# Assignment 6 — Custom Dataset and DataLoader

## Part A — Create Custom Dataset

- Create a class inheriting from: torch.utils.data.Dataset

- Dataset should contain:


| Hours Studied | Passed |
| :---: | :---: |
| 1 | 0 |
| 2 | 0 |
| 3 | 0 |
| 4 | 1 |
| 5 | 1 |
| 6 | 1 |

### Implement Required Method: 

- __init__ - Store features, labels

- __len__ - Return dataset size.

- __getitem__(idx)- Return: (x, y) for one sample.

In [22]:
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):

    def __init__(self):
        self.features=torch.tensor([[1],[2],[3],[4],[5],[6]],dtype=torch.float32)
        self.labels=torch.tensor([0,0,0,1,1,1])

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        x=self.features[index]
        y=self.labels[index]
        return x,y

## Part B — Create DataLoader

- Use: DataLoader

- Requirements: batch size = 2, shuffle = True

In [24]:
import torch
from torch.utils.data import DataLoader

dataset=CustomDataset()

# DataLoader
loader=DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

## Part C — Inspect Batches

Loop through dataloader and print:

- batch features
- batch labels


In [8]:
for batch_x,batch_y in loader:
    print(batch_x,batch_y)

tensor([[3.],
        [4.]]) tensor([0, 1])
tensor([[5.],
        [6.]]) tensor([1, 1])
tensor([[2.],
        [1.]]) tensor([0, 0])


# Part D — Build Binary Classifier

Architecture:

- Linear(1,8)
- ReLU
- Linear(8,2)

In [21]:
import torch
import torch.nn as nn

# Linear(1,8) -> ReLU -> #Linear(8,2)
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1=nn.Linear(1,8)
        self.relu=nn.ReLU()
        self.linear_2=nn.Linear(8,2)

    def forward(self,x):
        x=self.linear_1(x)
        x=self.relu(x)
        x=self.linear_2(x)
        return x
model=BinaryClassifier()

## Part E — Train Using Batches

- Inside training loop: use batches

- Train  using: CrossEntropyLoss, Adam

- Train for: 200 epochs

In [25]:
import torch
import torch.nn 
import torch.optim as optim

loss_fn=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.01)

# Training for 200 Epochs
for epoch in range(200):
    # Cummulative Loss for whole epoch
    epoch_loss=0
    for batch_x,batch_y in loader:
        predictions=model(batch_x)
        loss=loss_fn(predictions,batch_y)
        epoch_loss+=loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 10==0:
        print(f"Epoch {epoch}:", epoch_loss)


Epoch 0: 2.5015159249305725


Epoch 10: 1.6757270097732544
Epoch 20: 1.1698655784130096
Epoch 30: 0.8382255136966705
Epoch 40: 0.6225877702236176
Epoch 50: 0.4905345290899277
Epoch 60: 0.40971892327070236
Epoch 70: 0.3513154089450836
Epoch 80: 0.3048116434365511
Epoch 90: 0.2733587482944131
Epoch 100: 0.24455844797194004
Epoch 110: 0.22611676552332938
Epoch 120: 0.19561822339892387
Epoch 130: 0.18644424667581916
Epoch 140: 0.16095988149754703
Epoch 150: 0.15273328428156674
Epoch 160: 0.14143277541734278
Epoch 170: 0.12863695656415075
Epoch 180: 0.11906870396342129
Epoch 190: 0.10622082639019936


## Part F — Test Predictions

- Test: [[1],[3],[5]]

- Print predicted classes.

In [26]:
# Test Sample
test = torch.tensor([[1],[3],[5]], dtype=torch.float32)

with torch.no_grad():
    logits = model(test)
    preds = torch.argmax(logits, dim=1)

print(logits)
print(preds)

tensor([[ 4.0137, -3.5046],
        [ 1.6870, -0.4114],
        [-2.4614,  4.6446]])
tensor([0, 0, 1])
